## EDA

### 환경 설정

In [34]:
# 데이터 처리 및 분석
import pandas as pd
import ast
import numpy as np
from datetime import datetime, timedelta
import warnings
import re

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 통계 분석
from scipy import stats
from scipy.stats import shapiro, levene, ttest_ind, chi2_contingency, f_oneway
from scipy.stats import mannwhitneyu, fisher_exact, kruskal
import scikit_posthocs as sp
from statsmodels.stats.multicomp import pairwise_tukeyhsd, MultiComparison
import pingouin as pg
import platform

# ── 한글 폰트 설정 ──
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'

# ── 출력 설정 ──
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
np.random.seed(42)

# 시드 설정
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("=" * 60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


### 마스터테이블 병합 및 조인

#### 상품 정보 및 리뷰 데이터 전체 불러오기

In [35]:
# 데이터 로드
df_andar_product = pd.read_csv('./final_data/andar_products_nokids.csv')
df_andar_homepage_review = pd.read_csv('./final_data/andar_homepage_review_final_s2024.csv')
df_andar_musinsa_review = pd.read_csv('./final_data/andar_musinsa_review_final_s2024.csv')

df_fila_product = pd.read_csv('./final_data/fila_products_info_cleaned.csv')
df_fila_homepage_review = pd.read_csv('./final_data/fila_homepage_review_final_s2024.csv')
df_fila_musinsa_review = pd.read_csv('./final_data/fila_musinsa_review_final_s2024.csv')
df_fila_naver_review = pd.read_csv('./final_data/fila_naverstore_review_final_s2024.csv')

df_xexymix_product = pd.read_csv('./final_data/xexymix_product_info.csv')
df_xexymix_homepage_review = pd.read_csv('./final_data/xexymix_homepage_review_final_s2024.csv')
df_xexymix_musinsa_review = pd.read_csv('./final_data/xexymix_musinsa_review_final_s2024.csv')

df_lululemon_product = pd.read_csv('./final_data/lullu_product_info.csv')
df_lululemon_homepage_review = pd.read_csv('./final_data/lululemon_homepage_review_final_s2024.csv')
df_lululemon_musinsa_review = pd.read_csv('./final_data/lululemon_musinsa_review_final_s2024.csv')

print("데이터 로드 완료")
print(f"  안다르  - 상품정보: {len(df_andar_product):,}개 | 자사몰: {len(df_andar_homepage_review):,}건 | 무신사: {len(df_andar_musinsa_review):,}건")
print(f"  휠라    - 상품정보: {len(df_fila_product):,}개 | 자사몰: {len(df_fila_homepage_review):,}건 | 무신사: {len(df_fila_musinsa_review):,}건 | 네이버: {len(df_fila_naver_review):,}건")
print(f"  젝시믹스 - 상품정보: {len(df_xexymix_product):,}개 | 자사몰: {len(df_xexymix_homepage_review):,}건 | 무신사: {len(df_xexymix_musinsa_review):,}건")
print(f"  룰루레몬 - 상품정보: {len(df_lululemon_product):,}개 | 자사몰: {len(df_lululemon_homepage_review):,}건 | 무신사: {len(df_lululemon_musinsa_review):,}건")

데이터 로드 완료
  안다르  - 상품정보: 1,382개 | 자사몰: 638,989건 | 무신사: 6,376건
  휠라    - 상품정보: 569개 | 자사몰: 10,673건 | 무신사: 8,204건 | 네이버: 1,269건
  젝시믹스 - 상품정보: 1,432개 | 자사몰: 494,159건 | 무신사: 8,639건
  룰루레몬 - 상품정보: 935개 | 자사몰: 11,817건 | 무신사: 1,529건


### 안다르 리뷰데이터 병합 후 상품정보 조인

In [36]:
# ─── 상품정보 스키마 통일 ─────────────────────────────────────────────────
df_fila_product = df_fila_product.rename(columns={
    'category1': 'cat1', 'category2': 'cat2', 'category3': 'cat3'
})

PRODUCT_JOIN_COLS = ['product_id', 'brand', 'cat1', 'cat2', 'cat3',
                     'gender', 'original_price', 'discount_price',
                     'color_count', 'is_new', 'is_best']


# ─── 무신사/네이버 상품명 정규화 ──────────────────────────────────────────
def normalize_product_name(name, strip_brand_prefix=None):
    """
    무신사/네이버 상품명 정규화
    - 괄호·코드·색상코드 제거
    - strip_brand_prefix: 제거할 브랜드 접두어 (예: '휠라 ')
    """
    name = str(name)
    name = name.replace('Ⅰ', 'I').replace('Ⅱ', 'II').replace('Ⅲ', 'III')

    # 브랜드명 접두어 제거 (휠라 네이버: '휠라 레이 트레이너_...' → '레이 트레이너...')
    if strip_brand_prefix and name.startswith(strip_brand_prefix):
        name = name[len(strip_brand_prefix):]

    name = re.sub(r'\[[^\]]*\]', '', name)            # [2 PACK] 제거
    name = re.sub(r'\([^)]*\)', '', name)              # (8mm), (L) 제거
    name = re.sub(r'\s*/.*$', '', name)                # / 이후 전체 제거 (안다르: 상품명 / AMPPT-07)
    name = re.sub(r'[A-Z]{2,}[\d\-]+[A-Z\d]*', '', name)  # XT7104E, MMFJT-19 등 코드 제거
    name = re.sub(r'_', ' ', name)                     # _ 공백 치환 (코드 잔여물)
    # 끝에 남은 색상 약어(BLK, WHT, REDC 등 영문 3-4자) 제거
    name = re.sub(r'\b[A-Z]{3,5}\b', '', name)
    name = re.sub(r'\s+', ' ', name).strip()
    return name


def build_name_map(review_names, df_product, cutoff=0.7, strip_brand_prefix=None):
    """무신사/네이버 리뷰 상품명 → 상품정보 product_id 매핑 딕셔너리"""
    from difflib import get_close_matches

    prod_map = {
        normalize_product_name(r['product_name']): r['product_id']
        for _, r in df_product.iterrows()
    }
    prod_keys = list(prod_map.keys())

    name_to_id = {}
    unmatched = []

    for raw in review_names:
        norm = normalize_product_name(raw, strip_brand_prefix=strip_brand_prefix)

        # 1단계: 정확 매칭
        if norm in prod_map:
            name_to_id[raw] = prod_map[norm]; continue

        # 2단계: 접두어 매칭 (상품정보 이름이 리뷰 상품명의 앞부분)
        matched = next((pid for pk, pid in prod_map.items() if pk and norm.startswith(pk)), None)
        if matched:
            name_to_id[raw] = matched; continue

        # 3단계: 퍼지 매칭
        close = get_close_matches(norm, prod_keys, n=1, cutoff=cutoff)
        if close:
            name_to_id[raw] = prod_map[close[0]]
        else:
            unmatched.append(raw)

    match_cnt = len(name_to_id)
    total = len(review_names)
    print(f"  상품명 매칭: {match_cnt}/{total} ({match_cnt/total:.1%})")
    if unmatched:
        print(f"  미매칭 샘플: {unmatched[:3]}")

    return name_to_id


def join_other_channel(df_review, df_product, name_map):
    """무신사/네이버 리뷰 + 상품정보 LEFT JOIN (product_name 매칭 기반)"""
    prod_sub = df_product[PRODUCT_JOIN_COLS].rename(columns={'product_id': '_pid_info'})
    df = df_review.copy()
    df['_pid_info'] = df['product_name'].map(name_map)
    merged = df.merge(prod_sub, on='_pid_info', how='left').drop(columns=['_pid_info'])
    return merged


print("스키마 통일 및 헬퍼 함수 정의 완료")

스키마 통일 및 헬퍼 함수 정의 완료


#### 브랜드별 1차 마스터테이블 생성

In [37]:
# ─── 안다르 마스터테이블 ──────────────────────────────────────────────────
# 자사몰: product_id 직접 JOIN
andar_hp = df_andar_homepage_review.merge(
    df_andar_product[PRODUCT_JOIN_COLS], on='product_id', how='left'
)

# 무신사: product_name 정규화 매칭 후 JOIN
print('[안다르 무신사 상품명 매칭]')
andar_ms_map = build_name_map(df_andar_musinsa_review['product_name'].unique(), df_andar_product)
andar_ms = join_other_channel(df_andar_musinsa_review, df_andar_product, andar_ms_map)

df_andar_master = pd.concat([andar_hp, andar_ms], ignore_index=True)

print(f'\n안다르 마스터테이블: {len(df_andar_master):,}행')
print(f'  자사몰 {len(andar_hp):,}건 + 무신사 {len(andar_ms):,}건')
print(f'  상품정보 매칭률: {df_andar_master["brand"].notna().mean():.1%}')

[안다르 무신사 상품명 매칭]
  상품명 매칭: 181/181 (100.0%)

안다르 마스터테이블: 645,365행
  자사몰 638,989건 + 무신사 6,376건
  상품정보 매칭률: 98.8%


In [38]:
# ─── 휠라 마스터테이블 ────────────────────────────────────────────────────
fila_hp = df_fila_homepage_review.merge(
    df_fila_product[PRODUCT_JOIN_COLS], on='product_id', how='left'
)

# 무신사: cutoff=0.5 (에샤페 색상별 이름 차이 허용 — '에샤페 초코' ↔ '휠라 에샤페 LX 초코')
print('[휠라 무신사 상품명 매칭]')
fila_ms_map = build_name_map(
    df_fila_musinsa_review['product_name'].unique(), df_fila_product, cutoff=0.5
)
fila_ms = join_other_channel(df_fila_musinsa_review, df_fila_product, fila_ms_map)

# 네이버: '휠라 ' 접두어 제거 + cutoff=0.5
print('[휠라 네이버 상품명 매칭]')
fila_nv_map = build_name_map(
    df_fila_naver_review['product_name'].unique(), df_fila_product,
    cutoff=0.5, strip_brand_prefix='휠라 '
)
fila_nv = join_other_channel(df_fila_naver_review, df_fila_product, fila_nv_map)

df_fila_master = pd.concat([fila_hp, fila_ms, fila_nv], ignore_index=True)

print(f'\n휠라 마스터테이블: {len(df_fila_master):,}행')
print(f'  자사몰 {len(fila_hp):,}건 + 무신사 {len(fila_ms):,}건 + 네이버 {len(fila_nv):,}건')
print(f'  상품정보 매칭률: {df_fila_master["brand"].notna().mean():.1%}')

[휠라 무신사 상품명 매칭]
  상품명 매칭: 33/33 (100.0%)
[휠라 네이버 상품명 매칭]
  상품명 매칭: 327/363 (90.1%)
  미매칭 샘플: ['휠라 에샤페 7종 택1_FS261OD03X040 (실버문 베이지 블랙 크림 초코 말차 블루 핑크)', '휠라 에샤페 7종 택1_FS261OD03X040 (실버문 베이지 크림 말차 블랙 초코 핑크 블루)', '휠라 에샤페 6종 택1_FS253OD03X012 (초코 말차 카멜 브릭 블랙 모카)']

휠라 마스터테이블: 20,146행
  자사몰 10,673건 + 무신사 8,204건 + 네이버 1,269건
  상품정보 매칭률: 99.6%


In [39]:
# ─── 젝시믹스 마스터테이블 ────────────────────────────────────────────────
# 자사몰: product_id 직접 JOIN
xexymix_hp = df_xexymix_homepage_review.merge(
    df_xexymix_product[PRODUCT_JOIN_COLS], on='product_id', how='left'
)

# 무신사: product_name 매칭
print('[젝시믹스 무신사 상품명 매칭]')
xexymix_ms_map = build_name_map(df_xexymix_musinsa_review['product_name'].unique(), df_xexymix_product)
xexymix_ms = join_other_channel(df_xexymix_musinsa_review, df_xexymix_product, xexymix_ms_map)

df_xexymix_master = pd.concat([xexymix_hp, xexymix_ms], ignore_index=True)

print(f'\n젝시믹스 마스터테이블: {len(df_xexymix_master):,}행')
print(f'  자사몰 {len(xexymix_hp):,}건 + 무신사 {len(xexymix_ms):,}건')
print(f'  상품정보 매칭률: {df_xexymix_master["brand"].notna().mean():.1%}')

[젝시믹스 무신사 상품명 매칭]
  상품명 매칭: 706/867 (81.4%)
  미매칭 샘플: ['글로리아 라즈베리마젠타 XT4169T', '글로리아 할로겐블루 XT4169T', '글로리아 글루미네이비 XT4169T']

젝시믹스 마스터테이블: 502,798행
  자사몰 494,159건 + 무신사 8,639건
  상품정보 매칭률: 99.8%


In [40]:
# ─── 룰루레몬 마스터테이블 ───────────────────────────────────────────────
# 자사몰: product_id 직접 JOIN
lululemon_hp = df_lululemon_homepage_review.merge(
    df_lululemon_product[PRODUCT_JOIN_COLS], on='product_id', how='left'
)

# 무신사: product_name 매칭
print('[룰루레몬 무신사 상품명 매칭]')
lululemon_ms_map = build_name_map(df_lululemon_musinsa_review['product_name'].unique(), df_lululemon_product)
lululemon_ms = join_other_channel(df_lululemon_musinsa_review, df_lululemon_product, lululemon_ms_map)

df_lululemon_master = pd.concat([lululemon_hp, lululemon_ms], ignore_index=True)

print(f'\n룰루레몬 마스터테이블: {len(df_lululemon_master):,}행')
print(f'  자사몰 {len(lululemon_hp):,}건 + 무신사 {len(lululemon_ms):,}건')
print(f'  상품정보 매칭률: {df_lululemon_master["brand"].notna().mean():.1%}')

[룰루레몬 무신사 상품명 매칭]
  상품명 매칭: 445/469 (94.9%)
  미매칭 샘플: ['포플린 릴랙스 핏 팬츠  BLK', '인설레이티드 오버사이즈드 칼라드 재킷', '슬릭 풀집 러닝 재킷  BLK']

룰루레몬 마스터테이블: 13,346행
  자사몰 11,817건 + 무신사 1,529건
  상품정보 매칭률: 85.9%


#### 전체 마스터테이블 수직 병합

#### 무신사/네이버 미매칭 상품명 검증

In [41]:
from difflib import get_close_matches

def collect_unmatched(name_map, df_review, df_product, brand, channel,
                      strip_brand_prefix=None, top_n=3, cutoff_inspect=0.4):
    """미매칭 상품명을 브랜드/채널 정보와 함께 DataFrame으로 반환"""
    all_names = df_review['product_name'].unique()
    unmatched_raw = [n for n in all_names if n not in name_map]
    if not unmatched_raw:
        return pd.DataFrame()

    prod_norm_map = {
        normalize_product_name(r['product_name']): (r['product_id'], r['product_name'])
        for _, r in df_product.iterrows()
    }
    prod_norm_keys = list(prod_norm_map.keys())

    rows = []
    for raw in unmatched_raw:
        norm = normalize_product_name(raw, strip_brand_prefix=strip_brand_prefix)
        close_norms = get_close_matches(norm, prod_norm_keys, n=top_n, cutoff=cutoff_inspect)
        close_info = [prod_norm_map[k] for k in close_norms]
        cnt = (df_review['product_name'] == raw).sum()
        rows.append({
            'brand':           brand,
            'channel':         channel,
            '리뷰건수':         cnt,
            '리뷰_상품명':      raw,
            '정규화명':         norm,
            '추천_pid_1':      close_info[0][0] if len(close_info) > 0 else None,
            '추천_name_1':     close_info[0][1] if len(close_info) > 0 else '',
            '추천_pid_2':      close_info[1][0] if len(close_info) > 1 else None,
            '추천_name_2':     close_info[1][1] if len(close_info) > 1 else '',
            '추천_pid_3':      close_info[2][0] if len(close_info) > 2 else None,
            '추천_name_3':     close_info[2][1] if len(close_info) > 2 else '',
            '확정_product_id': '',   # ← 여기에 직접 채워서 돌려주세요
        })

    return pd.DataFrame(rows).sort_values('리뷰건수', ascending=False).reset_index(drop=True)


# ── 브랜드·채널별 미매칭 수집 ──
parts = []

parts.append(collect_unmatched(
    fila_nv_map,  df_fila_naver_review,    df_fila_product,
    brand='fila', channel='naver', strip_brand_prefix='휠라 '
))
parts.append(collect_unmatched(
    xexymix_ms_map, df_xexymix_musinsa_review, df_xexymix_product,
    brand='xexymix', channel='musinsa'
))
parts.append(collect_unmatched(
    lululemon_ms_map, df_lululemon_musinsa_review, df_lululemon_product,
    brand='lululemon', channel='musinsa'
))

df_unmatched_export = pd.concat(parts, ignore_index=True)
df_unmatched_export.to_csv('./final_data/unmatched_for_review.csv', index=False, encoding='utf-8-sig')

print(f"미매칭 목록 저장 완료: ./final_data/unmatched_for_review.csv")
print(f"총 {len(df_unmatched_export)}개 고유상품 / {df_unmatched_export['리뷰건수'].sum():,}건\n")
print(df_unmatched_export.groupby(['brand', 'channel'])[['리뷰건수']].agg(['count','sum']).rename(
    columns={'count': '고유상품수', 'sum': '총리뷰건수'}
))

미매칭 목록 저장 완료: ./final_data/unmatched_for_review.csv
총 221개 고유상품 / 1,046건

                   리뷰건수      
                  고유상품수 총리뷰건수
brand     channel            
fila      naver      36    83
lululemon musinsa    24    58
xexymix   musinsa   161   905


In [42]:
from difflib import get_close_matches

def inspect_unmatched(name_map, df_review, df_product, label, cutoff_inspect=0.4, max_show=30):
    """미매칭 상품명 원본·정규화·유사상품정보 나란히 출력"""
    all_names = df_review['product_name'].unique()
    unmatched_raw = [n for n in all_names if n not in name_map]

    if not unmatched_raw:
        print(f'[{label}] 미매칭 없음 ✓\n')
        return pd.DataFrame()

    prod_norm_map = {normalize_product_name(r['product_name']): r['product_name']
                     for _, r in df_product.iterrows()}
    prod_norm_keys = list(prod_norm_map.keys())

    rows = []
    for raw in unmatched_raw:
        norm = normalize_product_name(raw)
        close_norms = get_close_matches(norm, prod_norm_keys, n=3, cutoff=cutoff_inspect)
        close_orig = [prod_norm_map[k] for k in close_norms]
        # 해당 상품명 리뷰 건수
        cnt = (df_review['product_name'] == raw).sum()
        rows.append({
            '리뷰건수': cnt,
            '리뷰_상품명(원본)': raw,
            '정규화명': norm,
            '유사_상품정보(1위)': close_orig[0] if len(close_orig) > 0 else '',
            '유사_상품정보(2위)': close_orig[1] if len(close_orig) > 1 else '',
            '유사_상품정보(3위)': close_orig[2] if len(close_orig) > 2 else '',
        })

    df_result = pd.DataFrame(rows).sort_values('리뷰건수', ascending=False).reset_index(drop=True)

    print(f'[{label}] 미매칭 고유상품 {len(df_result)}개 '
          f'(리뷰 {df_result["리뷰건수"].sum():,}건)')
    display(df_result.head(max_show))
    return df_result


# ── 안다르 ──
unmatched_andar_ms = inspect_unmatched(
    andar_ms_map, df_andar_musinsa_review, df_andar_product, '안다르 무신사'
)

# ── 휠라 ──
unmatched_fila_ms = inspect_unmatched(
    fila_ms_map, df_fila_musinsa_review, df_fila_product, '휠라 무신사'
)
unmatched_fila_nv = inspect_unmatched(
    fila_nv_map, df_fila_naver_review, df_fila_product, '휠라 네이버'
)

# ── 젝시믹스 ──
unmatched_xexymix_ms = inspect_unmatched(
    xexymix_ms_map, df_xexymix_musinsa_review, df_xexymix_product, '젝시믹스 무신사'
)

# ── 룰루레몬 ──
unmatched_lululemon_ms = inspect_unmatched(
    lululemon_ms_map, df_lululemon_musinsa_review, df_lululemon_product, '룰루레몬 무신사'
)

[안다르 무신사] 미매칭 없음 ✓

[휠라 무신사] 미매칭 없음 ✓

[휠라 네이버] 미매칭 고유상품 36개 (리뷰 83건)


,리뷰건수,리뷰_상품명(원본),정규화명,유사_상품정보(1위),유사_상품정보(2위),유사_상품정보(3위)
0,17,휠라 OAKMONT TR v2_1JM02786H067,휠라 OAKMONT TR v2 1,휠라 에버그랜드 TR v3,휠라 웨이비 데이 v2,휠라 에샤페 v2 실버
1,11,휠라 에샤페 7종 택1_FS261OD03X040 (실버문 베이지 블랙 크림 초코 말...,휠라 에샤페 7종 택1,휠라 에샤페 v2 실버,휠라 에샤페 MS,휠라 에샤페 LX 크림
2,5,휠라 체크셔츠_FS253SH01X001541,휠라 체크셔츠,휠라 스포츠 힙색,휠라 글리오 핑크,휠라 글리오 크림
3,5,휠라 푸퍼 버킷백 4종 택1_FS254RB01F004,휠라 푸퍼 버킷백 4종 택1,휠라 레비오 04/26 LX,휠라 에버그랜드 TR v3,휠라 스포르트 4부 레깅스
4,5,휠라 OAKMONT TR v2_1JM02786H924,휠라 OAKMONT TR v2 1,휠라 에버그랜드 TR v3,휠라 웨이비 데이 v2,휠라 에샤페 v2 실버
5,4,휠라 에샤페 7종 택1_FS261OD03X040 (실버문 베이지 크림 말차 블랙 초...,휠라 에샤페 7종 택1,휠라 에샤페 v2 실버,휠라 에샤페 MS,휠라 에샤페 LX 크림
6,3,휠라 푸퍼 숄더백 2종 택1_FS254RB01F001,휠라 푸퍼 숄더백 2종 택1,휠라 플로트 맥스 2.0 쉴드,휠라 플로트 맥스 2.0 LT,휠라 웨이비 데이 v2
7,3,휠라 에샤페 6종 택1_FS253OD03X012 (초코 말차 카멜 브릭 블랙 모카),휠라 에샤페 6종 택1,휠라 에샤페 v2 실버,휠라 에샤페 MS,휠라 에샤페 LX 크림
8,2,휠라 FILA WAVY DAY 3종 택1_1RM02850H,휠라 3종 택1 1,휠라 웨이비 데이 v2,휠라 에샤페 v2 실버,휠라 에샤페 LX 크림
9,2,휠라플러스 리니어 크레스트 V넥 스웨터_FO253ST10X003650,휠라플러스 리니어 크레스트 V넥 스웨터,FHC 헤리티지 크루넥 스웨터,휠라 리트모 슬릭 LX,휠라 플레르 VC


[젝시믹스 무신사] 미매칭 고유상품 161개 (리뷰 905건)


,리뷰건수,리뷰_상품명(원본),정규화명,유사_상품정보(1위),유사_상품정보(2위),유사_상품정보(3위)
0,216,젤라 인텐션 부츠컷 팬츠 블랙 XWFLG02H3_XWFLG06J1,젤라 인텐션 부츠컷 팬츠 블랙,젤라 인텐션 바이커 쇼츠 5부,[2PACK] 젤라 인텐션 바이커 쇼츠 3부,트윌텐션 포멀 부츠컷 팬츠
1,84,젤라 인텐션 부츠컷 팬츠 제트차콜 XWFLG02H3_XWFLG06J1,젤라 인텐션 부츠컷 팬츠 제트차콜,젤라 인텐션 바이커 쇼츠 5부,[2PACK] 젤라 인텐션 바이커 쇼츠 3부,트윌텐션 포멀 부츠컷 팬츠
2,57,에어드로즈 투인원 쇼츠 XP0108T,에어드로즈 투인원 쇼츠,에어라이트 셔링 투인원 쇼츠,RX 에어라이트 투인원 쇼츠,[덱스PICK] RX 맨즈 에어로 메쉬 투인원 쇼츠
3,28,젤라 인텐션 부츠컷 팬츠 토네이도그레이 XWFLG02H3_XWFLG06J1,젤라 인텐션 부츠컷 팬츠 토네이도그레이,젤라 인텐션 7.5부 레깅스,젤라 인텐션 바이커 쇼츠 5부,[2PACK] 젤라 인텐션 바이커 쇼츠 3부
4,26,리브드 텐션 슬리브리스 블랙 XFK2TE1101,리브드 텐션 슬리브리스 블랙,리브드 폴로 숏슬리브,RX 그리드 메쉬 슬리브리스,쿨브리즈 맨즈 슬리브리스
5,26,리브드 텐션 슬리브리스 백아이보리 XFK2TE1101,리브드 텐션 슬리브리스 백아이보리,리브드 폴로 숏슬리브,RX 그리드 메쉬 슬리브리스,쿨브리즈 맨즈 슬리브리스
6,20,데일리 서포트 하프 레깅스 블랙 XP2105F,데일리 서포트 하프 레깅스 블랙,컴포트파인 카프리 레깅스,NX 맨즈 서포트 레깅스 3.5부,데일리 하프문 새들백
7,19,젤라 인텐션 부츠컷 팬츠 클라우드그레이 XWFLG02H3_XWFLG06J1,젤라 인텐션 부츠컷 팬츠 클라우드그레이,젤라 인텐션 7.5부 레깅스,젤라 인텐션 바이커 쇼츠 5부,[2PACK] 젤라 인텐션 바이커 쇼츠 3부
8,15,라이트텐션 스트레이트 팬츠 XFK3PT1006,라이트텐션 스트레이트 팬츠,라이트 스트레치 부츠컷 팬츠,라이트 랩 스커트 쇼츠,라이트텐션 맨즈 카고 조거팬츠
9,14,젤라 인텐션 부츠컷 팬츠 우드스모크 XWFLG02H3_XWFLG06J1,젤라 인텐션 부츠컷 팬츠 우드스모크,젤라 인텐션 7.5부 레깅스,젤라 인텐션 바이커 쇼츠 5부,[2PACK] 젤라 인텐션 바이커 쇼츠 3부


[룰루레몬 무신사] 미매칭 고유상품 24개 (리뷰 58건)


,리뷰건수,리뷰_상품명(원본),정규화명,유사_상품정보(1위),유사_상품정보(2위),유사_상품정보(3위)
0,16,포플린 릴랙스 핏 팬츠 BLK,포플린 릴랙스 핏 팬츠,밸런서 릴랙스 핏 팬츠 *레귤러,"비캄 릴랙스 핏 쇼츠 7""","밸런서 릴랙스 핏 쇼츠 7"""
1,5,슬릭 풀집 러닝 재킷 BLK,슬릭 풀집 러닝 재킷,슬릭 시티 롱 재킷,슬릭 시티 재킷,원더 트레인 풀집 재킷
2,5,인설레이티드 오버사이즈드 칼라드 재킷,인설레이티드 오버사이즈드 칼라드 재킷,인설레이티드 유틸리티 셔츠 재킷,인설레이티드 백 벤트 러닝 재킷,UV 프로텍티브 오버사이즈드 테니스 재킷
3,3,러브 롱슬리브 셔츠 *슬럽 저지 DVGY,러브 롱슬리브 셔츠 *슬럽 저지,스위프틀리 울 롱슬리브 셔츠 *힙 렝스,홀드 타이트 롱슬리브 셔츠 *온라인 한정,이지셋 숏슬리브 셔츠 *워시
4,3,페이스 라이벌 하이라이즈 쇼츠 5inch BLK,페이스 라이벌 하이라이즈 쇼츠 5inch,"댄스 스튜디오 하이라이즈 쇼츠 3.5""","스위프트 스피드 하이라이즈 쇼츠 6""","겟 로우 하이라이즈 트레이닝 쇼츠 5"""
5,3,원더 트레인 글러브 VITP BLK,원더 트레인 글러브,원더 트레인 풀집 재킷,원더 트레인 패디드 글러브 *유니섹스 사이즈,원더 트레인 크롭 롱슬리브 하프집
6,2,라운지풀 배럴 레그 하이라이즈 크롭 팬츠 MINK,라운지풀 배럴 레그 하이라이즈 크롭 팬츠,"원더 트레인 하이라이즈 크롭 21""",그루브 와이드 레그 하이라이즈 팬츠 *레귤러,"라이선스 투 트레인 하이라이즈 쇼츠 4"""
7,2,슬릭 풀집 러닝 재킷 REDC,슬릭 풀집 러닝 재킷,슬릭 시티 롱 재킷,슬릭 시티 재킷,원더 트레인 풀집 재킷
8,2,제로드 인 크링클 텍스처 하프집 SILD,제로드 인 크링클 텍스처 하프집,제로드 인 탱크,제로드 인 슬림 핏 팬츠,원더 트레인 크롭 롱슬리브 하프집
9,2,저지 헨리 베이스 레이어 REDC SQUO,저지 헨리 베이스 레이어,모크넥 롱슬리브 골프 베이스 레이어,모크넥 골프 롱슬리브 베이스 레이어,페이스 브레이커 팬츠


In [43]:
# ─── 수동 보정 매핑 ────────────────────────────────────────────────────────
#
# 잔여 미매칭 요약:
#   · 안다르 무신사   : 0건   (완전 해소)
#   · 휠라 무신사     : 0건   (완전 해소)
#   · 휠라 네이버     : 83건  (묶음상품 '7종 택1' 등 → 단일 product_id 매핑 불가)
#   · 젝시믹스 무신사 : 905건 (상품정보 크롤링 누락 상품 → 상품정보 없음)
#   · 룰루레몬 무신사 : 58건  (상품정보 크롤링 누락 상품 → 상품정보 없음)
#   확인 후 매핑 가능한 항목이 있다면 아래 형식으로 추가.
#   형식: { '리뷰_상품명(원본 그대로)': product_id }

MANUAL_MAP = {
    # 예시 (실제 매핑이 확인된 경우에만 추가):
    # '슬릭 풀집 러닝 재킷  BLK': 'prod20001819',   # 룰루레몬: 슬릭 시티 롱 재킷과 동일 상품이면 추가
}


def apply_manual_map(df_master, df_product, manual_map):
    """MANUAL_MAP 기반으로 brand 컬럼이 NaN인 행 재보정"""
    if not manual_map:
        return df_master

    prod_lookup = df_product.set_index('product_id')[
        [c for c in PRODUCT_JOIN_COLS if c != 'product_id']
    ].to_dict('index')

    df = df_master.copy()
    fixed = 0
    for review_name, pid in manual_map.items():
        if pid not in prod_lookup:
            print(f'  경고: product_id {pid} 를 상품정보에서 찾을 수 없음'); continue
        mask = (df['product_name'] == review_name) & df['brand'].isna()
        for col, val in prod_lookup[pid].items():
            df.loc[mask, col] = val
        fixed += mask.sum()

    if fixed:
        print(f"수동 보정 완료: {fixed:,}건 업데이트")
    return df


df_andar_master    = apply_manual_map(df_andar_master,    df_andar_product,    MANUAL_MAP)
df_fila_master     = apply_manual_map(df_fila_master,     df_fila_product,     MANUAL_MAP)
df_xexymix_master  = apply_manual_map(df_xexymix_master,  df_xexymix_product,  MANUAL_MAP)
df_lululemon_master= apply_manual_map(df_lululemon_master, df_lululemon_product, MANUAL_MAP)

print("수동 보정 적용 완료 (MANUAL_MAP이 비어있으면 변경 없음)")

수동 보정 적용 완료 (MANUAL_MAP이 비어있으면 변경 없음)


In [44]:
# ─── 전체 마스터테이블 수직 병합 ───
df_master = pd.concat([
    df_andar_master,
    df_fila_master,
    df_xexymix_master,
    df_lululemon_master,
], ignore_index=True)

print(f'전체 마스터테이블: {len(df_master):,}행 × {len(df_master.columns)}컬럼')
print(f'컬럼 목록: {list(df_master.columns)}')
print()
print('[브랜드별 리뷰 수]')
print(df_master['brand'].value_counts(dropna=False).to_string())
print()
print('[채널별 리뷰 수]')
print(df_master['platform'].value_counts(dropna=False).to_string())
print()
print(f'전체 상품정보 매칭률: {df_master["brand"].notna().mean():.1%}')

# 저장
df_master.to_csv('./final_data/master_table.csv', index=False, encoding='utf-8-sig')
print('\n마스터테이블 저장 완료: ./final_data/master_table.csv')

전체 마스터테이블: 1,181,655행 × 31컬럼
컬럼 목록: ['collect_date', 'platform', 'review_id', 'product_id', 'product_name', 'review_date', 'year', 'month', 'content', 'rating', 'helpful_count', 'has_image', 'purchase_option', 'purchase_option_color', 'purchase_option_size', 'women_size_yn', 'user_height', 'user_weight', 'user_height_group', 'user_weight_group', 'product_url', 'brand', 'cat1', 'cat2', 'cat3', 'gender', 'original_price', 'discount_price', 'color_count', 'is_new', 'is_best']

[브랜드별 리뷰 수]
brand
안다르     637529
젝시믹스    501893
FILA     20063
룰루레몬     11460
NaN      10710

[채널별 리뷰 수]
platform
자사몰       1155638
무신사         24748
네이버스토어       1269

전체 상품정보 매칭률: 99.1%

마스터테이블 저장 완료: ./final_data/master_table.csv
